In [1]:
# Building familiriaty with optimisation tools in python

In [2]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.optimize import Bounds
from scipy.optimize import LinearConstraint
import cvxpy as cp
import plotly.express as px

In [3]:
# Minimise simple function f(x) = (x-3)^2 + 2
def f(x):
    """Simple function to test minimisation"""
    return ((x-3)**2.0 + 2)

x0 = 0
bounds = Bounds(lb=5)

res = minimize(f, x0, bounds=bounds)
#options={'disp':True})
print(res.x)
print(res.fun)

[5.]
6.0


In [4]:
# Minmise f(x_1,x_2) = -(p_1.x_1 + p_2.x_2)
# where
# x_1,x_2 are enger discharged in period 1 and period 2
#p1=50, p2= 100 (£/MWh)
# subject to: x_1 + x_2 <= 1 (battery capacity of 1 MWh)
# And: x_1, x_2 >= 0

def f2(x):
    """Trival battery discharge problem"""
    return - (50*x[0] + 100*x[1])

x0 = [0,0]
A = np.array([1, 1])
lb = 0
ub = 1
bounds = Bounds([0,0], [np.inf, np.inf])

linear_constraint = LinearConstraint(A,lb,ub)
result = minimize(f2, x0, bounds=bounds, constraints=linear_constraint)
print(result)




     message: Optimization terminated successfully
     success: True
      status: 0
         fun: -99.99999997708073
           x: [ 0.000e+00  1.000e+00]
         nit: 2
         jac: [-5.000e+01 -1.000e+02]
        nfev: 6
        njev: 2
 multipliers: [ 0.000e+00  1.000e+02]


In [5]:
# Solve trivial battery dispatch problem in cvxpy
x = cp.Variable(2)


# Constraints
constraints = [ x[0] >= 0,
                x[1] >= 0,
                x[0] + x[1] <=1]

#Objective
obj = cp.Minimize(-(50*x[0] + 100*x[1]))

#Form & solve problem
prob = cp.Problem(obj, constraints)
prob.solve() #Returns the optimal vlaue
print("status", prob.status)
print("optimal value", prob.value)
print("optimal var", x[0].value, x[1].value)


status optimal
optimal value -99.99999923400327
optimal var 5.293068918983694e-09 0.9999999896934982


In [6]:
# Extend battery dispatch problem to 48 periods
# s: Storage capacity [MWh]
# c: Charge rate [MW]
# d: Discharge rate [MW]
# prices: Electricity prices [£/MWh]
# t: time granularity [h] 
# eta: effiency [eta]

n = 48
t = 0.5
eta = 0.9
periods = np.arange(n)
prices = (
    20 + 
    30 * np.exp(-((periods - 14)**2) / 10) +  # morning peak around period 14 (7am)
    50 * np.exp(-((periods - 34)**2) / 10)     # evening peak around period 34 (5pm)
)
print(len(prices))

s = cp.Variable(n)
d = cp.Variable(n)
c = cp.Variable(n)

s_start = 0.5
s_end = 0.5

constraints2 = [ 0 <= c, c <= 0.2,
                0 <=d,  d <= 0.2,
                0 <=s, s <= 1,
                s[0] == s_start,
                s[n-1] == s_end]

# Add time linking contraints
# s_i = s_{i-1} + 0.5 * (c_i - d_i) for i = 2...n
constraints2.append(
    s[1:n-1] == s[0:n-2] + t * (eta*c[1:n-1] - d[1:n-1])
)

#Objective
obj2 = cp.Maximize(t*prices@(d-c))

#Form & solve problem
prob2 = cp.Problem(obj2, constraints2)
prob2.solve() #Returns the optimal vlaue
print("status", prob2.status)
print("optimal value", prob2.value)

# Plot optimal values
df = pd.DataFrame({
    'Time': range(1, n+1),
    'Prices':prices,
    'Storage level':s.value,
    'Charge rate': c.value,
    'Discharge rate':d.value
})

fig_s = px.line(df, x='Time', y='Storage level', 
                title='Storage level over time')
fig_prices = px.line(df, x='Time', y='Prices', 
                title='Prices over time')
fig_cd = px.line(df, x='Time', y=['Charge rate', 'Discharge rate'], 
                title='Charge/Discharge rate over time')

fig_s.show()
fig_cd.show()
fig_prices.show()
        
    


48
status optimal
optimal value 53.76977959551023


In [7]:
# Do practice portfolio optimisation problem
# See https://docs.mosek.com/portfolio-cookbook/markowitz.html

# Synthetic data set-up
np.random.seed(42)
n = 5  # assets
mu = np.array([0.10, 0.12, 0.08, 0.15, 0.09])  # expected returns
# Random covariance matrix (positive semi-definite)
A = np.random.randn(n, n)
Sigma = A.T @ A / n + 0.01 * np.eye(n)
r_min = 0.10  # minimum required return


w = cp.Variable(n)


constraints3 = [ 0 <= w,
                mu.T@w >= r_min,
                cp.sum(w) == 1]
                

#Objective
obj3 = cp.Minimize(w.T@Sigma@w)

#Form & solve problem
prob3 = cp.Problem(obj3, constraints3)
prob3.solve() #Returns the optimal vlaue
print("status", prob3.status)
print("optimal value", prob3.value)
print("w", w.value)


status optimal
optimal value 0.1624454415467822
w [ 2.67057345e-01  1.38346989e-01  4.64238901e-01  1.30356765e-01
 -3.25963487e-22]


In [8]:
# Electric fleet charging example problem

# h = 0.5 [h] granularity
# v_[t,i]: Vehicle i state of charge at time t [kWh]
# c_[t,i]: Vehicle i charge rate at time t [kW]
# prices_t: Electricity price at time t [£/kWh] 
# c_max: Maximum charging rate per vehicle
# grid: Grid connection limit [kW]
# v_max: Vehical battery capacity [kWh]
# tarrive_i: arrival time for vehicle i
# v0_i: arrival state of charge for vehicle i 

# Constraints
# v_[0,i] = v0_i #For simplicity take vehicle state of charge to be v0 at all times before t_arrive
# v_[11,i] >= 0.8*v_max #Each vehicle must reach 80% charge by 6am 
# c_[t,i] <= c_max
# v_[t,i] <= v_max
# c_[t,i] == 0 for all t<=t_arrive
# sum_i(c_[t,i]) <= grid
# v_[t,i] = v[t-1,i] + h*c[t,i]

# Objective function
# Minimise cost
# Minimise (h*sum_t{sum_i(c_[t,i])*prices_t}

h = 0.5 #granularity
T = 48 # number of time steps
I = 10 # number of vehicles
c_max = 7 # max charge rate [kW]
v_max = 40 # max vehicle battery capacity [kWh]
grid = 50 # grid connection limit [kw]

# Generate arrival state of charge between 20-50% full
rng = np.random.default_rng(seed=42)
v0 = (rng.random(I)*0.3+0.2)*v_max

# Generate arrival times
t_arrive = rng.integers(low=0, high=7, size=10)

v = cp.Variable((T,I))
c = cp.Variable((T,I))

constraints4 = [ v[0,:] == v0,
                 v[11,:] >= 0.8*v_max,
                0 <=c,  c <= c_max,
                0 <=v, v <= v_max,
                cp.sum(c,axis=1)<=50]

# Create zero mask for restricting charging before t_arrive
time_indices = np.arange(T)[:, None]
zero_mask = time_indices<t_arrive
constraints4.append(
    c[zero_mask] == 0
)

# Time linking of state of charge
constraints4.append(
    v[1:T-1,:] == v[0:T-2,:]+h*c[1:T-1,:]
)

#Objective function

# Sum over vehicles
total_per_timestep = cp.sum(c,axis=1)

obj4 = cp.Minimize(h*total_per_timestep@prices)

#Form & solve problem
prob4 = cp.Problem(obj4, constraints4)
prob4.solve(solver=cp.HIGHS) #Returns the optimal vlaue
print("status", prob4.status)
print("optimal value", prob4.value)

vehicle_output = v.value
charge_output = c.value

# Convert the matrix into a Pandas DataFrame
# Each column represents a vehicle, each row is a time step
df = pd.DataFrame(
    vehicle_output, 
    columns=[f"Vehicle {i}" for i in range(I)]
)

df["Time"] = np.arange(T)

fig_veh = px.line(
    df, 
    x="Time", 
    y=[f"Vehicle {i}" for i in range(I)], # Selects all vehicle columns to plot
    labels={"value": "Vehicle state of charge (v)", "variable": "Vehicles"},
    title="Vehicle state of charge"
)

# Display the interactive chart
fig_veh.show()





status optimal
optimal value 3466.7063855292163


In [9]:
print(vehicle_output[11,:])

charge_output = c.value
total_charge_output = np.sum(charge_output,axis=1)

# Convert the matrix into a Pandas DataFrame
# Each column represents a vehicle, each row is a time step
df2 = pd.DataFrame(
    charge_output, 
    columns=[f"Vehicle {i}" for i in range(I)]
)

df2["Time"] = np.arange(T)
df2["Total charge"] = total_charge_output

fig_totc = px.line(df2, x="Time", y="Total charge")
fig_totc.show()

fig_c = px.line(
    df2, 
    x="Time", 
    y=[f"Vehicle {i}" for i in range(I)], # Selects all vehicle columns to plot
    labels={"value": "Vehicle charge rate (c)", "variable": "Vehicles"},
    title="Vehicle charge rate"
)

# Display the interactive chart
fig_c.show()
print("T arrive:", t_arrive)

[32. 32. 32. 32. 32. 32. 32. 32. 32. 32.]


T arrive: [3 2 1 6 5 4 2 5 3 3]


In [10]:
#Unit Commitment problem

# Three generators
# g1_i: generator 1 production [MW] at time period i
# g2_i: generator 2 production [MW] at time period i
# g3_i: generator 3 production [MW] at time period i
# b1_i: binary variable for when generator 1 on at time period i
# b2_i: binary variable for when generator 2 on at time period i
# b3_i: binary variable for when generator 3 on at time period i
# c1: production cost of generator 1 [£/MWh]
# c2: production cost of generator 2 [£/MWh]
# c3: production cost of generator 3 [£/MWh]
# sc1: start-up cost of generator 1 [£]
# sc2: start-up cost of generator 2 [£]
# sc3: start-up cost of generator 3 [£]
# s1_i: Counts number of starts of generator 1 - is 1 if generator 1 starts-up in period  i, 0 otherwise
# s2_i: Counts number of starts of generator 2 - is 1 if generator 2 starts-up in period  i, 0 otherwise
# s3_i: Counts number of starts of generator 3 - is 1 if generator 3 starts-up in period  i, 0 otherwise


# d_i : Demand at time period [i]
# t: time granularity [h]

# Objective function (Cost equation)
# sum_i{ [g1_i*c1 + g2_i*c2 + g3_i*c3]*t + (s1_i*sc1 + s2_i*sc2 + s3_i*sc3)}

# Count starts
# s1_i >=0, s1_i >= b1_i - b1_(i-1)
# s2_i >=0, s2_i >= b2_i - b2_(i-1)
# s3_i>=0, s3_i >= b3_i - b3_(i-1)
# Boundary condition of start counting
# s1_0 >= b1_0
# s2_0 >= b2_0
# s3_0 >= b3_0

# Constraints
# Supply-demand balance
# g1_i + g2_i + g3_i == d_i

# Min output of generator
# g1_i >= 50*b1_i
# g2_i >= 20*b1_i
# g3_i >= 10*b1_i

# Max output of generator
# g1_i <= 200
# g2_i <= 100
# g3_i <= 50


t = 0.5 #granularity
I = 12 # number of time steps
g1_max = 200 # max g1 output [MW]
g2_max = 100 # max g2 output [MW]
g3_max = 50 # max g3 output [MW]
g1_min = 50 # min g1 output [MW]
g2_min = 20 #min g2 output [MW]
g3_min = 10 #min g3 output [MW]

# Costs
c1 = 20
c2 = 35
c3 = 55
sc1 = 100
sc2 = 50
sc3 = 20

demand = np.array([120, 130, 150, 180, 200, 210, 200, 180, 160, 140, 120, 110])

g1 = cp.Variable(I)
b1 = cp.Variable(I, boolean=True)
s1 = cp.Variable(I)

g2 = cp.Variable(I)
b2 = cp.Variable(I, boolean=True)
s2 = cp.Variable(I)

g3 = cp.Variable(I)
b3 = cp.Variable(I, boolean=True)
s3 = cp.Variable(I)



constraints5 = [ g1+ g2 +g3 == demand, g1<= g1_max*b1,
                 g2<= g2_max*b2,
                 g3<=g3_max*b3,
                 g1>=g1_min*b1,
                 g2>=g2_min*b2,
                 g3>=g3_min*b3,
                 s1>=0,
                 s2>=0,
                 s3>=0,
                s1[0] >= b1[0],
                s2[0] >= b2[0],
                s3[0] >= b3[0],
                s1[1:I-1] >= b1[1:I-1] - b1[0:I-2],
                s2[1:I-1] >= b2[1:I-1] - b2[0:I-2],
                s3[1:I-1] >= b3[1:I-1] - b3[0:I-2]]


#Objective function
obj5 = cp.Minimize(cp.sum(g1*c1 + g2*c2 + g3*c3)*t + cp.sum(s1*sc1 + s2*sc2 + s3*sc3))

#Form & solve problem
prob5 = cp.Problem(obj5, constraints5)
prob5.solve(solver=cp.HIGHS) #Returns the optimal vlaue
print("status", prob5.status)
print("optimal value", prob5.value)

g1_output = g1.value
g2_output = g2.value
g3_output = g3.value
total_gen = g1_output + g2_output + g3_output

# Convert the matrix into a Pandas DataFrame
# Each column represents a vehicle, each row is a time step
df = pd.DataFrame({
    'Time': np.arange(I),
    'demand': demand,
    'gen1': g1_output,
    'gen2': g2_output,
    'gen3': g3_output,
    'total gen': total_gen
})

fig_gen = px.line(
    df, 
    x="Time", 
    y=["gen1", "gen2","gen3"],
    labels={"value": "Generation", "variable": "Generators"},
    title="Plant generation"
)

fig_dem = px.line(
    df, x="Time", y = ["demand", "total gen"], title = "Generation vs Demand")

# Display charts
fig_gen.show()
fig_dem.show()

print(s1.value)
print(s2.value)
print(s3.value)

#print(b1.value)
#print(b2.value)
#print(b3.value)



status optimal
optimal value 19294.99999905


[ 1. -0. -0.  0. -0. -0. -0. -0. -0. -0. -0. -0.]
[ 0.00000000e+00  0.00000000e+00 -0.00000000e+00 -0.00000000e+00
 -0.00000000e+00  1.00000022e-09 -0.00000000e+00 -0.00000000e+00
 -0.00000000e+00  0.00000000e+00  0.00000000e+00 -0.00000000e+00]
[ 0.  0. -0. -0. -0.  1. -0. -0. -0.  0.  0. -0.]
